# Journal Extension J4 — Statistical Significance on the Original 83.95% Test Set

## Purpose

This notebook adds the statistical evidence required for a strong Q1/Q2 journal submission.

It works on the **exact original 80/10/10 thesis protocol**:

- Validation: 2,533
- Internal test: 2,534
- Final proposed model: `NoTTA_StackingBlend_ClassBias`
- Final internal test BACC: about **83.95%**

No training is performed.

## Analyses

### 1. 95% confidence intervals
Using **5,000 class-stratified bootstrap replicates** for:

- Accuracy
- Balanced Accuracy (BACC)
- Macro-F1

Models:
- Final proposed ensemble
- SimCLR No-TTA
- DINOv2 ViT-S/14 3-seed mean
- Rotation
- Optimized Weighted Ensemble

### 2. Paired bootstrap differences
For the frozen final model versus:

- SimCLR
- DINOv2
- Rotation
- Optimized Weighted Ensemble

For:
- ΔBACC
- ΔMacro-F1
- ΔAccuracy

Reports:
- mean observed difference
- 95% CI
- bootstrap probability `P(Final > Comparator)`
- two-sided bootstrap p-value

### 3. Exact McNemar tests
Paired correctness comparison between the final model and each comparator.

Reports:
- both correct
- final-only correct
- comparator-only correct
- both wrong
- exact two-sided McNemar p-value

### 4. Multiple-comparison correction
Holm correction is applied across the main McNemar comparisons.

## Required Kaggle inputs

Attach:

1. **Original full thesis pipeline output** (`last-part-of-the-previous-one`)  
   Needed for SimCLR and Rotation probabilities.

2. **Confidence-gated ensemble output**  
   Needed for the frozen final and optimized weighted ensemble probabilities.

3. **J2 DINOv2 output**  
   Needed for DINO same-split probabilities.

4. **J1 output** is recommended for protocol verification, but not mandatory.

## No images and no GPU are required.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / CONFIG
# ============================================================
import json
import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

from scipy.stats import binomtest

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

OUTPUT_DIR = WORKING / "j4_original_split_statistics"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
AUDIT_DIR = OUTPUT_DIR / "audit"

for d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]
NUM_CLASSES = 8

EXPECTED_VAL_N = 2533
EXPECTED_TEST_N = 2534

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42

print("=" * 96)
print("J4 — ORIGINAL-SPLIT STATISTICAL SIGNIFICANCE")
print("=" * 96)
print("Bootstrap replicates:", N_BOOTSTRAP)
print("Bootstrap seed      :", BOOTSTRAP_SEED)
print("Output              :", OUTPUT_DIR)

J4 — ORIGINAL-SPLIT STATISTICAL SIGNIFICANCE
Bootstrap replicates: 5000
Bootstrap seed      : 42
Output              : /kaggle/working/j4_original_split_statistics


In [2]:
# ============================================================
# CELL 2 — HELPERS
# ============================================================

def find_all(filename):
    return sorted(
        KAGGLE_INPUT.rglob(filename),
        key=lambda p: (len(p.parts), str(p))
    )

def normalize_probabilities(probabilities):
    p = np.asarray(probabilities, dtype=np.float64)

    if p.ndim != 2 or p.shape[1] != NUM_CLASSES:
        raise ValueError(
            f"Expected probability shape (N,{NUM_CLASSES}); got {p.shape}"
        )

    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p = np.clip(p, 1e-12, None)
    p /= p.sum(axis=1, keepdims=True)

    return p.astype(np.float32)

def probability_columns(df):
    cols = [
        c for c in df.columns
        if c.startswith("prob_class_")
        and c.rsplit("_", 1)[-1].isdigit()
    ]
    if len(cols) == NUM_CLASSES:
        return sorted(cols, key=lambda c: int(c.rsplit("_", 1)[-1]))

    cols = [
        c for c in df.columns
        if c.startswith("prob_")
        and len(c.split("_")) >= 2
        and c.split("_")[1].isdigit()
    ]
    if len(cols) == NUM_CLASSES:
        return sorted(cols, key=lambda c: int(c.split("_")[1]))

    exact = [
        f"prob_{i}"
        for i in range(NUM_CLASSES)
        if f"prob_{i}" in df.columns
    ]
    if len(exact) == NUM_CLASSES:
        return exact

    raise ValueError(
        "Could not detect exactly 8 probability columns."
    )

def label_column(df):
    for c in ["y_true", "true_label", "label"]:
        if c in df.columns:
            return c
    raise ValueError("True-label column not found.")

def load_prediction_csv(path):
    df = pd.read_csv(path)
    y_col = label_column(df)
    p_cols = probability_columns(df)

    y = df[y_col].to_numpy(np.int32)
    p = normalize_probabilities(df[p_cols].to_numpy(np.float32))

    return y, p

def metric_triplet(y_true, probabilities):
    pred = np.argmax(probabilities, axis=1)

    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "bacc": float(balanced_accuracy_score(y_true, pred)),
        "macro_f1": float(
            f1_score(
                y_true,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
    }

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=np.float64)
    m = len(p_values)

    order = np.argsort(p_values)
    adjusted = np.empty(m, dtype=np.float64)

    running_max = 0.0

    for rank, idx in enumerate(order):
        raw_adjusted = (m - rank) * p_values[idx]
        running_max = max(running_max, raw_adjusted)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted

print("Helpers ready.")

Helpers ready.


In [3]:
# ============================================================
# CELL 3 — LOAD FROZEN FINAL / WEIGHTED ENSEMBLE
# ============================================================

final_npz_hits = find_all("final_internal_probabilities.npz")

ranked = sorted(
    final_npz_hits,
    key=lambda p: (
        0 if "reuse_only_all_model_ensemble" in str(p) else
        1 if "j1_original_protocol_lock" in str(p) else
        9,
        len(p.parts),
        str(p),
    )
)

FINAL_NPZ = None

for candidate in ranked:
    try:
        with np.load(candidate) as z:
            if not {
                "y_test",
                "final_test_prob",
                "weighted_test_prob",
            }.issubset(z.files):
                continue

            if len(z["y_test"]) != EXPECTED_TEST_N:
                continue

        FINAL_NPZ = candidate
        break
    except Exception:
        continue

if FINAL_NPZ is None:
    raise FileNotFoundError(
        "Original confidence-gated final_internal_probabilities.npz "
        "with 2,534 test samples was not found."
    )

with np.load(FINAL_NPZ) as z:
    Y_TEST = np.asarray(z["y_test"], dtype=np.int32)
    FINAL_TEST = normalize_probabilities(z["final_test_prob"])
    WEIGHTED_TEST = normalize_probabilities(z["weighted_test_prob"])

if len(Y_TEST) != EXPECTED_TEST_N:
    raise RuntimeError("Wrong test size.")

print("Frozen final NPZ:")
print(FINAL_NPZ)
print("Test shape:", Y_TEST.shape)

Frozen final NPZ:
/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/final_internal_probabilities.npz
Test shape: (2534,)


In [4]:
# ============================================================
# CELL 4 — FIND ORIGINAL PIPELINE ROOT + SIMCLR / ROTATION
# ============================================================

sim_hits = find_all("multiseed_probabilities_no_tta.npz")

PIPELINE_ROOT = None
SIMCLR_NPZ = None

for candidate in sim_hits:
    try:
        with np.load(candidate) as z:
            if not {"y_test", "test_prob"}.issubset(z.files):
                continue

            yt = np.asarray(z["y_test"], dtype=np.int32)

            if len(yt) != EXPECTED_TEST_N:
                continue

            if not np.array_equal(yt, Y_TEST):
                continue

        SIMCLR_NPZ = candidate
        PIPELINE_ROOT = candidate.parent.parent
        break

    except Exception:
        continue

if PIPELINE_ROOT is None:
    raise FileNotFoundError(
        "Could not identify original thesis pipeline root."
    )

with np.load(SIMCLR_NPZ) as z:
    SIMCLR_TEST = normalize_probabilities(z["test_prob"])

ROTATION_TEST_CSV = (
    PIPELINE_ROOT
    / "extended_imagenet_rotation_finetune"
    / "imagenet_rotation_test_predictions.csv"
)

if not ROTATION_TEST_CSV.exists():
    raise FileNotFoundError(str(ROTATION_TEST_CSV))

rotation_y, ROTATION_TEST = load_prediction_csv(
    ROTATION_TEST_CSV
)

if not np.array_equal(rotation_y, Y_TEST):
    raise RuntimeError(
        "Rotation test labels do not align with frozen final labels."
    )

print("Pipeline root:")
print(PIPELINE_ROOT)
print("SimCLR :", SIMCLR_NPZ)
print("Rotation:", ROTATION_TEST_CSV)

Pipeline root:
/kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike
SimCLR : /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/simclr_finetune_multiseed_bacc_v2/multiseed_probabilities_no_tta.npz
Rotation: /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/extended_imagenet_rotation_finetune/imagenet_rotation_test_predictions.csv


In [5]:
# ============================================================
# CELL 5 — LOAD J2 DINO SAME-SPLIT PROBABILITIES
# ============================================================

dino_hits = find_all(
    "dinov2_original_split_multiseed_probabilities.npz"
)

DINO_NPZ = None

for candidate in dino_hits:
    try:
        with np.load(candidate) as z:
            if not {"y_test", "test_prob"}.issubset(z.files):
                continue

            yt = np.asarray(z["y_test"], dtype=np.int32)

            if not np.array_equal(yt, Y_TEST):
                continue

        DINO_NPZ = candidate
        break

    except Exception:
        continue

if DINO_NPZ is None:
    raise FileNotFoundError(
        "J2 DINO same-split probability file not found."
    )

with np.load(DINO_NPZ) as z:
    DINO_TEST = normalize_probabilities(z["test_prob"])

print("DINO:")
print(DINO_NPZ)

DINO:
/kaggle/input/notebooks/anirbanroysudipto/journal-extension-j2/j2_dinov2_original_split/dinov2_original_split_multiseed_probabilities.npz


In [6]:
# ============================================================
# CELL 6 — OBSERVED TEST METRICS
# ============================================================

MODELS = {
    "Final Proposed": FINAL_TEST,
    "SimCLR No-TTA": SIMCLR_TEST,
    "DINOv2 ViT-S/14": DINO_TEST,
    "Rotation": ROTATION_TEST,
    "Optimized Weighted Ensemble": WEIGHTED_TEST,
}

observed_rows = []

for name, p in MODELS.items():
    m = metric_triplet(Y_TEST, p)

    observed_rows.append({
        "model": name,
        **m,
    })

OBSERVED_DF = pd.DataFrame(observed_rows)
display(OBSERVED_DF)

OBSERVED_DF.to_csv(
    TABLE_DIR / "observed_internal_test_metrics.csv",
    index=False,
)

print("\nObserved final BACC:")
print(
    f"{100 * OBSERVED_DF.loc[OBSERVED_DF['model']=='Final Proposed','bacc'].iloc[0]:.4f}%"
)

                         model  accuracy      bacc  macro_f1
0               Final Proposed  0.838990  0.839549  0.797961
1                SimCLR No-TTA  0.844515  0.810729  0.814429
2              DINOv2 ViT-S/14  0.796369  0.804301  0.788338
3                     Rotation  0.795185  0.737443  0.737314
4  Optimized Weighted Ensemble  0.850829  0.810289  0.812911


Observed final BACC:
83.9549%


In [7]:
# ============================================================
# CELL 7 — CLASS-STRATIFIED PAIRED BOOTSTRAP INDEX GENERATOR
# ============================================================

rng = np.random.default_rng(BOOTSTRAP_SEED)

class_indices = {
    c: np.flatnonzero(Y_TEST == c)
    for c in range(NUM_CLASSES)
}

for c in range(NUM_CLASSES):
    if len(class_indices[c]) == 0:
        raise RuntimeError(f"Class {c} absent from test set.")

def draw_stratified_indices():
    pieces = []

    for c in range(NUM_CLASSES):
        idx = class_indices[c]

        sampled = rng.choice(
            idx,
            size=len(idx),
            replace=True,
        )

        pieces.append(sampled)

    return np.concatenate(pieces)

# Smoke test
_test_idx = draw_stratified_indices()
assert len(_test_idx) == len(Y_TEST)

print("Per-class test counts:")
for c, name in enumerate(CLASS_NAMES):
    print(name, len(class_indices[c]))

print("\nStratified bootstrap generator: PASSED")

Per-class test counts:
MEL 452
NV 1288
BCC 333
AK 86
BKL 263
DF 24
VASC 25
SCC 63

Stratified bootstrap generator: PASSED


In [8]:
# ============================================================
# CELL 8 — 5,000 BOOTSTRAP REPLICATES FOR EACH MODEL
# ============================================================

metric_names = [
    "accuracy",
    "bacc",
    "macro_f1",
]

bootstrap_store = {
    model_name: {
        metric: np.empty(N_BOOTSTRAP, dtype=np.float32)
        for metric in metric_names
    }
    for model_name in MODELS
}

# Store bootstrap indices implicitly by calculating all models
# on the same sampled rows within each replicate.
for b in range(N_BOOTSTRAP):
    idx = draw_stratified_indices()
    yb = Y_TEST[idx]

    for model_name, p in MODELS.items():
        mb = metric_triplet(
            yb,
            p[idx],
        )

        for metric in metric_names:
            bootstrap_store[model_name][metric][b] = mb[metric]

    if (b + 1) % 1000 == 0:
        print(f"Bootstrap {b+1}/{N_BOOTSTRAP}")

print("Bootstrap complete.")

Bootstrap 1000/5000
Bootstrap 2000/5000
Bootstrap 3000/5000
Bootstrap 4000/5000
Bootstrap 5000/5000
Bootstrap complete.


In [9]:
# ============================================================
# CELL 9 — 95% CONFIDENCE INTERVAL TABLE
# ============================================================

ci_rows = []

for model_name in MODELS:
    observed = metric_triplet(
        Y_TEST,
        MODELS[model_name],
    )

    for metric in metric_names:
        values = bootstrap_store[
            model_name
        ][metric]

        low, high = np.percentile(
            values,
            [2.5, 97.5],
        )

        ci_rows.append({
            "model": model_name,
            "metric": metric,
            "observed": observed[metric],
            "bootstrap_mean": float(np.mean(values)),
            "ci95_low": float(low),
            "ci95_high": float(high),
            "n_bootstrap": N_BOOTSTRAP,
            "bootstrap_type": "class-stratified paired bootstrap",
        })

CI_DF = pd.DataFrame(ci_rows)
display(CI_DF)

CI_DF.to_csv(
    TABLE_DIR / "bootstrap_95ci_all_models.csv",
    index=False,
)

# Publication-friendly percent table
CI_PERCENT_DF = CI_DF.copy()

for c in [
    "observed",
    "bootstrap_mean",
    "ci95_low",
    "ci95_high",
]:
    CI_PERCENT_DF[c] = 100 * CI_PERCENT_DF[c]

CI_PERCENT_DF.to_csv(
    TABLE_DIR / "bootstrap_95ci_all_models_percent.csv",
    index=False,
)

display(CI_PERCENT_DF)

                          model    metric  observed  bootstrap_mean  ci95_low  \
0                Final Proposed  accuracy  0.838990        0.838998  0.825178   
1                Final Proposed      bacc  0.839549        0.839537  0.811958   
2                Final Proposed  macro_f1  0.797961        0.798231  0.772103   
3                 SimCLR No-TTA  accuracy  0.844515        0.844494  0.830702   
4                 SimCLR No-TTA      bacc  0.810729        0.810597  0.779843   
5                 SimCLR No-TTA  macro_f1  0.814429        0.814012  0.787262   
6               DINOv2 ViT-S/14  accuracy  0.796369        0.796250  0.780584   
7               DINOv2 ViT-S/14      bacc  0.804301        0.804060  0.775142   
8               DINOv2 ViT-S/14  macro_f1  0.788338        0.787995  0.763996   
9                      Rotation  accuracy  0.795185        0.795055  0.779795   
10                     Rotation      bacc  0.737443        0.737263  0.702525   
11                     Rotat

                          model    metric   observed  bootstrap_mean  \
0                Final Proposed  accuracy  83.898974       83.899754   
1                Final Proposed      bacc  83.954912       83.953708   
2                Final Proposed  macro_f1  79.796095       79.823089   
3                 SimCLR No-TTA  accuracy  84.451460       84.449434   
4                 SimCLR No-TTA      bacc  81.072854       81.059736   
5                 SimCLR No-TTA  macro_f1  81.442907       81.401199   
6               DINOv2 ViT-S/14  accuracy  79.636938       79.625028   
7               DINOv2 ViT-S/14      bacc  80.430113       80.405957   
8               DINOv2 ViT-S/14  macro_f1  78.833811       78.799474   
9                      Rotation  accuracy  79.518548       79.505491   
10                     Rotation      bacc  73.744272       73.726267   
11                     Rotation  macro_f1  73.731393       73.691565   
12  Optimized Weighted Ensemble  accuracy  85.082873       85.08

In [10]:
# ============================================================
# CELL 10 — PAIRED BOOTSTRAP DIFFERENCES: FINAL VS COMPARATORS
# ============================================================

COMPARATORS = [
    "SimCLR No-TTA",
    "DINOv2 ViT-S/14",
    "Rotation",
    "Optimized Weighted Ensemble",
]

paired_rows = []

final_observed = metric_triplet(
    Y_TEST,
    FINAL_TEST,
)

for comparator in COMPARATORS:
    comparator_observed = metric_triplet(
        Y_TEST,
        MODELS[comparator],
    )

    for metric in metric_names:
        diff = (
            bootstrap_store["Final Proposed"][metric]
            - bootstrap_store[comparator][metric]
        )

        low, high = np.percentile(
            diff,
            [2.5, 97.5],
        )

        prob_positive = float(
            np.mean(diff > 0)
        )

        prob_nonpositive = float(
            np.mean(diff <= 0)
        )

        # Two-sided empirical bootstrap p-value.
        # Add 1 to avoid exact zero with finite Monte Carlo samples.
        count_nonpos = int(np.sum(diff <= 0))
        count_nonneg = int(np.sum(diff >= 0))

        p_two = min(
            1.0,
            2.0 * min(
                (count_nonpos + 1) / (N_BOOTSTRAP + 1),
                (count_nonneg + 1) / (N_BOOTSTRAP + 1),
            ),
        )

        paired_rows.append({
            "comparison": f"Final Proposed - {comparator}",
            "metric": metric,
            "observed_difference": (
                final_observed[metric]
                - comparator_observed[metric]
            ),
            "bootstrap_mean_difference": float(np.mean(diff)),
            "ci95_low": float(low),
            "ci95_high": float(high),
            "prob_final_greater": prob_positive,
            "two_sided_bootstrap_p": float(p_two),
            "n_bootstrap": N_BOOTSTRAP,
        })

PAIRED_BOOT_DF = pd.DataFrame(
    paired_rows
)

display(PAIRED_BOOT_DF)

PAIRED_BOOT_DF.to_csv(
    TABLE_DIR / "paired_bootstrap_final_vs_comparators.csv",
    index=False,
)

PAIRED_BOOT_PERCENT_DF = PAIRED_BOOT_DF.copy()

for c in [
    "observed_difference",
    "bootstrap_mean_difference",
    "ci95_low",
    "ci95_high",
]:
    PAIRED_BOOT_PERCENT_DF[c] = (
        100 * PAIRED_BOOT_PERCENT_DF[c]
    )

PAIRED_BOOT_PERCENT_DF.to_csv(
    TABLE_DIR / "paired_bootstrap_final_vs_comparators_percent.csv",
    index=False,
)

display(PAIRED_BOOT_PERCENT_DF)

                                      comparison    metric  \
0                 Final Proposed - SimCLR No-TTA  accuracy   
1                 Final Proposed - SimCLR No-TTA      bacc   
2                 Final Proposed - SimCLR No-TTA  macro_f1   
3               Final Proposed - DINOv2 ViT-S/14  accuracy   
4               Final Proposed - DINOv2 ViT-S/14      bacc   
5               Final Proposed - DINOv2 ViT-S/14  macro_f1   
6                      Final Proposed - Rotation  accuracy   
7                      Final Proposed - Rotation      bacc   
8                      Final Proposed - Rotation  macro_f1   
9   Final Proposed - Optimized Weighted Ensemble  accuracy   
10  Final Proposed - Optimized Weighted Ensemble      bacc   
11  Final Proposed - Optimized Weighted Ensemble  macro_f1   

    observed_difference  bootstrap_mean_difference  ci95_low  ci95_high  \
0             -0.005525                  -0.005497 -0.013812   0.003157   
1              0.028821                   0

                                      comparison    metric  \
0                 Final Proposed - SimCLR No-TTA  accuracy   
1                 Final Proposed - SimCLR No-TTA      bacc   
2                 Final Proposed - SimCLR No-TTA  macro_f1   
3               Final Proposed - DINOv2 ViT-S/14  accuracy   
4               Final Proposed - DINOv2 ViT-S/14      bacc   
5               Final Proposed - DINOv2 ViT-S/14  macro_f1   
6                      Final Proposed - Rotation  accuracy   
7                      Final Proposed - Rotation      bacc   
8                      Final Proposed - Rotation  macro_f1   
9   Final Proposed - Optimized Weighted Ensemble  accuracy   
10  Final Proposed - Optimized Weighted Ensemble      bacc   
11  Final Proposed - Optimized Weighted Ensemble  macro_f1   

    observed_difference  bootstrap_mean_difference  ci95_low  ci95_high  \
0             -0.552486                  -0.549676 -1.381218   0.315708   
1              2.882058                   2

In [11]:
# ============================================================
# CELL 11 — EXACT MCNEMAR TESTS ON OVERALL CORRECTNESS
# ============================================================

final_pred = np.argmax(
    FINAL_TEST,
    axis=1,
)

final_correct = (
    final_pred == Y_TEST
)

mcnemar_rows = []

for comparator in COMPARATORS:
    comp_pred = np.argmax(
        MODELS[comparator],
        axis=1,
    )

    comp_correct = (
        comp_pred == Y_TEST
    )

    both_correct = int(
        np.sum(
            final_correct
            & comp_correct
        )
    )

    final_only = int(
        np.sum(
            final_correct
            & ~comp_correct
        )
    )

    comparator_only = int(
        np.sum(
            ~final_correct
            & comp_correct
        )
    )

    both_wrong = int(
        np.sum(
            ~final_correct
            & ~comp_correct
        )
    )

    discordant = (
        final_only
        + comparator_only
    )

    if discordant == 0:
        p_value = 1.0
    else:
        p_value = float(
            binomtest(
                k=min(
                    final_only,
                    comparator_only,
                ),
                n=discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )

    mcnemar_rows.append({
        "comparison": f"Final Proposed vs {comparator}",
        "both_correct": both_correct,
        "final_only_correct": final_only,
        "comparator_only_correct": comparator_only,
        "both_wrong": both_wrong,
        "discordant_pairs": discordant,
        "mcnemar_exact_p": p_value,
    })

MCNEMAR_DF = pd.DataFrame(
    mcnemar_rows
)

MCNEMAR_DF[
    "holm_adjusted_p"
] = holm_adjust(
    MCNEMAR_DF[
        "mcnemar_exact_p"
    ].to_numpy()
)

MCNEMAR_DF[
    "significant_after_holm_0.05"
] = (
    MCNEMAR_DF[
        "holm_adjusted_p"
    ] < 0.05
)

display(MCNEMAR_DF)

MCNEMAR_DF.to_csv(
    TABLE_DIR / "mcnemar_exact_final_vs_comparators.csv",
    index=False,
)

                                      comparison  both_correct  \
0                Final Proposed vs SimCLR No-TTA          2072   
1              Final Proposed vs DINOv2 ViT-S/14          1866   
2                     Final Proposed vs Rotation          1898   
3  Final Proposed vs Optimized Weighted Ensemble          2083   

   final_only_correct  comparator_only_correct  both_wrong  discordant_pairs  \
0                  54                       68         340               122   
1                 260                      152         256               412   
2                 228                      117         291               345   
3                  43                       73         335               116   

   mcnemar_exact_p  holm_adjusted_p  significant_after_holm_0.05  
0     2.390888e-01     2.390888e-01                        False  
1     1.152990e-07     3.458969e-07                         True  
2     2.331752e-09     9.327009e-09                         True  


In [12]:
# ============================================================
# CELL 12 — BACC DIFFERENCE FIGURE
# ============================================================

bacc_diff_df = PAIRED_BOOT_DF.loc[
    PAIRED_BOOT_DF["metric"] == "bacc"
].copy()

labels = [
    x.replace(
        "Final Proposed - ",
        ""
    )
    for x in bacc_diff_df["comparison"]
]

means = (
    100
    * bacc_diff_df[
        "observed_difference"
    ].to_numpy()
)

lower = (
    means
    - 100
    * bacc_diff_df[
        "ci95_low"
    ].to_numpy()
)

upper = (
    100
    * bacc_diff_df[
        "ci95_high"
    ].to_numpy()
    - means
)

plt.figure(
    figsize=(9.0, 5.5)
)

x = np.arange(
    len(labels)
)

plt.errorbar(
    x,
    means,
    yerr=np.vstack([lower, upper]),
    fmt="o",
    capsize=5,
)

plt.axhline(
    0.0,
    linewidth=1,
)

plt.xticks(
    x,
    labels,
    rotation=20,
    ha="right",
)

plt.ylabel(
    "Δ Balanced Accuracy (percentage points)"
)

plt.title(
    "Paired Bootstrap: Final Proposed Model vs Comparators"
)

plt.grid(
    axis="y",
    alpha=0.2
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "paired_bootstrap_bacc_differences.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [13]:
# ============================================================
# CELL 13 — FINAL MODEL CI FIGURE
# ============================================================

final_ci = CI_PERCENT_DF.loc[
    CI_PERCENT_DF[
        "model"
    ] == "Final Proposed"
].copy()

metric_label_map = {
    "accuracy": "Accuracy",
    "bacc": "Balanced Accuracy",
    "macro_f1": "Macro-F1",
}

labels = [
    metric_label_map[m]
    for m in final_ci["metric"]
]

obs = final_ci[
    "observed"
].to_numpy()

low = final_ci[
    "ci95_low"
].to_numpy()

high = final_ci[
    "ci95_high"
].to_numpy()

plt.figure(
    figsize=(7.6, 5.2)
)

x = np.arange(
    len(labels)
)

plt.errorbar(
    x,
    obs,
    yerr=np.vstack([
        obs - low,
        high - obs,
    ]),
    fmt="o",
    capsize=6,
)

plt.xticks(
    x,
    labels,
)

plt.ylabel(
    "Performance (%)"
)

plt.title(
    "Final Proposed Model — 95% Bootstrap Confidence Intervals"
)

plt.grid(
    axis="y",
    alpha=0.2
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "final_model_bootstrap_95ci.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [14]:
# ============================================================
# CELL 14 — JOURNAL INTERPRETATION SUMMARY
# ============================================================

def row_for_pair(comparator, metric="bacc"):
    return PAIRED_BOOT_DF.loc[
        (
            PAIRED_BOOT_DF["comparison"]
            == f"Final Proposed - {comparator}"
        )
        &
        (
            PAIRED_BOOT_DF["metric"]
            == metric
        )
    ].iloc[0]

summary = {
    "protocol": "original stratified 80/10/10",
    "test_n": len(Y_TEST),
    "bootstrap_replicates": N_BOOTSTRAP,
    "bootstrap_type": "class-stratified paired bootstrap",
    "bootstrap_seed": BOOTSTRAP_SEED,
    "final_observed_metrics": metric_triplet(
        Y_TEST,
        FINAL_TEST,
    ),
    "final_vs_simclr_bacc": row_for_pair(
        "SimCLR No-TTA",
        "bacc",
    ).to_dict(),
    "final_vs_dinov2_bacc": row_for_pair(
        "DINOv2 ViT-S/14",
        "bacc",
    ).to_dict(),
    "mcnemar": MCNEMAR_DF.to_dict(
        orient="records"
    ),
    "interpretation_rule": (
        "Primary effect-size evidence is the paired bootstrap difference "
        "and its 95% CI. McNemar evaluates paired overall correctness, "
        "which is not identical to balanced accuracy."
    ),
}

with open(
    AUDIT_DIR / "J4_statistical_summary.json",
    "w",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        default=float,
    )

print(
    json.dumps(
        summary,
        indent=2,
        default=float,
    )
)

{
  "protocol": "original stratified 80/10/10",
  "test_n": 2534,
  "bootstrap_replicates": 5000,
  "bootstrap_type": "class-stratified paired bootstrap",
  "bootstrap_seed": 42,
  "final_observed_metrics": {
    "accuracy": 0.8389897395422258,
    "bacc": 0.8395491150909984,
    "macro_f1": 0.7979609543641538
  },
  "final_vs_simclr_bacc": {
    "comparison": "Final Proposed - SimCLR No-TTA",
    "metric": "bacc",
    "observed_difference": 0.028820579808856883,
    "bootstrap_mean_difference": 0.028939813375473022,
    "ci95_low": 0.007946383953094483,
    "ci95_high": 0.05220508277416229,
    "prob_final_greater": 0.9986,
    "two_sided_bootstrap_p": 0.003199360127974405,
    "n_bootstrap": 5000
  },
  "final_vs_dinov2_bacc": {
    "comparison": "Final Proposed - DINOv2 ViT-S/14",
    "metric": "bacc",
    "observed_difference": 0.03524798492676495,
    "bootstrap_mean_difference": 0.035477619618177414,
    "ci95_low": 0.00848977118730545,
    "ci95_high": 0.0624030604958534,
    "p

In [15]:
# ============================================================
# CELL 15 — COMPLETION MARKER + ZIP
# ============================================================

final_metrics = metric_triplet(
    Y_TEST,
    FINAL_TEST,
)

marker = {
    "stage": "J4_original_split_statistics",
    "status": "complete",
    "protocol": "original_stratified_80_10_10",
    "test_n": len(Y_TEST),
    "bootstrap_replicates": N_BOOTSTRAP,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "final_test_accuracy": final_metrics["accuracy"],
    "final_test_bacc": final_metrics["bacc"],
    "final_test_macro_f1": final_metrics["macro_f1"],
    "test_used_for_selection": False,
    "output_rel": "j4_original_split_statistics",
}

MARKER_PATH = WORKING / "J4_STATISTICS_COMPLETE.json"

with open(
    MARKER_PATH,
    "w",
) as f:
    json.dump(
        marker,
        f,
        indent=2,
    )

shutil.copy2(
    MARKER_PATH,
    OUTPUT_DIR / "J4_STATISTICS_COMPLETE.json",
)

ZIP_PATH = WORKING / "J4_ORIGINAL_SPLIT_STATISTICS_RESULTS.zip"

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for p in sorted(
        OUTPUT_DIR.rglob("*")
    ):
        if p.is_file():
            zf.write(
                p,
                arcname=str(
                    p.relative_to(
                        OUTPUT_DIR.parent
                    )
                ),
            )

required = [
    TABLE_DIR / "bootstrap_95ci_all_models.csv",
    TABLE_DIR / "paired_bootstrap_final_vs_comparators.csv",
    TABLE_DIR / "mcnemar_exact_final_vs_comparators.csv",
    FIG_DIR / "paired_bootstrap_bacc_differences.png",
    FIG_DIR / "final_model_bootstrap_95ci.png",
    AUDIT_DIR / "J4_statistical_summary.json",
    MARKER_PATH,
    ZIP_PATH,
]

missing = [
    str(p)
    for p in required
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing J4 outputs:\n"
        + "\n".join(missing)
    )

print("=" * 96)
print("J4 STATISTICAL ANALYSIS COMPLETE")
print("=" * 96)

print(
    "Final Accuracy:",
    f"{100 * final_metrics['accuracy']:.4f}%"
)

print(
    "Final BACC    :",
    f"{100 * final_metrics['bacc']:.4f}%"
)

print(
    "Final Macro-F1:",
    f"{100 * final_metrics['macro_f1']:.4f}%"
)

print(
    "Bootstrap replicates:",
    N_BOOTSTRAP
)

print(
    "ZIP:",
    ZIP_PATH
)

print()
print(
    "Save this Kaggle notebook Version WITH OUTPUTS "
    "and send the executed notebook back."
)

J4 STATISTICAL ANALYSIS COMPLETE
Final Accuracy: 83.8990%
Final BACC    : 83.9549%
Final Macro-F1: 79.7961%
Bootstrap replicates: 5000
ZIP: /kaggle/working/J4_ORIGINAL_SPLIT_STATISTICS_RESULTS.zip

Save this Kaggle notebook Version WITH OUTPUTS and send the executed notebook back.
